# 📊 mBART PHINC — Machine Translation Evaluation

Evaluates the fine-tuned **mBART-large-50** model on the **PHINC** (Hinglish → English) test set.

### Metrics computed
| Metric | Library | What it measures |
|--------|---------|------------------|
| **BLEU** | `sacrebleu` | n-gram precision (standard MT metric) |
| **chrF / chrF++** | `sacrebleu` | Character n-gram F-score |
| **ROUGE-1/2/L** | `rouge_score` | Recall-oriented overlap |
| **BERTScore** | `bert_score` | Contextual embedding similarity |
| **TER** | `sacrebleu` | Translation edit rate (lower = better) |

---

## 1. Install Dependencies

In [ ]:
!pip install -q sacrebleu rouge_score bert_score datasets transformers sentencepiece protobuf

## 2. Mount Google Drive & Set Paths

In [ ]:
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    base_dir = "/content/drive/MyDrive/ML_Project/mbart-phinc"
    print("✅ Google Drive mounted.")
except ImportError:
    base_dir = "./mbart-phinc"
    print("ℹ️  Running locally — using local directories.")

# ── Path to your saved final model ──────────────────────────────────────────
MODEL_PATH = os.path.join(base_dir, "checkpoint-2319")

# ── (Optional) path to saved tokenized test split ───────────────────────────
TOKENIZED_TEST_PATH = "/content/drive/MyDrive/shared/phinc/mbart/tokenized_data/test"

print(f"Model path : {MODEL_PATH}")
print(f"Model exists: {os.path.exists(MODEL_PATH)}")

In [ ]:
print(f"Listing contents of MODEL_PATH: {MODEL_PATH}")
!ls -F "{MODEL_PATH}"

In [ ]:
print(f"Listing contents of base_dir: {base_dir}")
!ls -F "{base_dir}"

## 3. Load Model & Tokenizer

In [ ]:
import torch
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast
import os # Import os for path existence check

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

print("Loading tokenizer & model …")

# The previous check for MODEL_PATH existence will be removed.
# The transformers library's error message will be more precise.

tokenizer = MBart50TokenizerFast.from_pretrained(MODEL_PATH)
model     = MBartForConditionalGeneration.from_pretrained(MODEL_PATH).to(device)
model.eval()

# mBART language codes: Hinglish input treated as Hindi (hi_IN)
tokenizer.src_lang = "hi_IN"
TARGET_LANG_ID = tokenizer.lang_code_to_id["en_XX"]

print("✅ Model loaded.")

## 4. Load Test Data

In [ ]:
from datasets import load_from_disk

# ── Load the tokenized test split saved during training ──────────────────────
# This avoids any re-download or re-tokenization.
# Columns present: input_ids, attention_mask, labels
print(f"Loading tokenized test split from:\n  {TOKENIZED_TEST_PATH}")
test_dataset = load_from_disk(TOKENIZED_TEST_PATH)
print(f"✅ Loaded {len(test_dataset)} examples.")
print(f"   Columns: {test_dataset.column_names}")

# ── Recover reference strings by decoding the saved label token IDs ──────────
# Labels were stored as token IDs with -100 padding; replace -100 with pad_token_id before decoding.
import numpy as np

label_ids = test_dataset["labels"]   # list of lists
cleaned_labels = [
    [tok if tok != -100 else tokenizer.pad_token_id for tok in ids]
    for ids in label_ids
]
references_all = tokenizer.batch_decode(cleaned_labels, skip_special_tokens=True)

print("\nSample decoded references:")
for i in range(3):
    print(f"  [{i}] {references_all[i]}")

## 5. Generate Translations

> Adjust `EVAL_SAMPLES` to control how many examples to evaluate.
> Set to `None` to run on the full test set (slow without GPU).

In [ ]:
import torch
from tqdm.auto import tqdm

# ── Config ───────────────────────────────────────────────────────────────────
EVAL_SAMPLES = None  # None = full test set; set e.g. 500 to run a subset
BATCH_SIZE   = 16
MAX_LENGTH   = 128
NUM_BEAMS    = 4

# ── Slice ────────────────────────────────────────────────────────────────────
eval_dataset = test_dataset.select(range(EVAL_SAMPLES)) if EVAL_SAMPLES else test_dataset
references   = references_all[:len(eval_dataset)]
n            = len(eval_dataset)
print(f"Evaluating on {n} examples …")

# ── Batch generation using saved input_ids (no re-tokenization needed) ───────
hypotheses = []

for i in tqdm(range(0, n, BATCH_SIZE), desc="Translating"):
    batch = eval_dataset.select(range(i, min(i + BATCH_SIZE, n)))

    # Pad input_ids within batch
    input_ids = torch.nn.utils.rnn.pad_sequence(
        [torch.tensor(ids) for ids in batch["input_ids"]],
        batch_first=True,
        padding_value=tokenizer.pad_token_id
    ).to(device)

    attention_mask = (input_ids != tokenizer.pad_token_id).long()

    with torch.no_grad():
        generated = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            forced_bos_token_id=TARGET_LANG_ID,
            num_beams=NUM_BEAMS,
            max_length=MAX_LENGTH,
            early_stopping=True
        )

    decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
    hypotheses.extend(decoded)

# ── Decode source sentences from saved input_ids for display ─────────────────
sources = tokenizer.batch_decode(
    [ids for ids in eval_dataset["input_ids"]], skip_special_tokens=True
)

print(f"\n✅ Generated {len(hypotheses)} translations.")
print("\nSample predictions:")
for i in range(5):
    print(f"  SRC  : {sources[i]}")
    print(f"  HYP  : {hypotheses[i]}")
    print(f"  REF  : {references[i]}")
    print()

## 6. Compute Evaluation Metrics

### 6.1 BLEU, chrF, chrF++ and TER (sacrebleu)

In [ ]:
import sacrebleu

# sacrebleu expects list-of-references as [[ref1, ref2, ...]] (one list per reference set)
refs_wrapped = [references]   # single reference per example

# BLEU
bleu_score = sacrebleu.corpus_bleu(hypotheses, refs_wrapped)
print(f"BLEU        : {bleu_score.score:.2f}")

# chrF
chrf_score  = sacrebleu.corpus_chrf(hypotheses, refs_wrapped)
print(f"chrF        : {chrf_score.score:.2f}")

# chrF++ (word order sensitive)
chrfpp_score = sacrebleu.corpus_chrf(hypotheses, refs_wrapped, word_order=2)
print(f"chrF++      : {chrfpp_score.score:.2f}")

# TER (lower is better)
ter_score   = sacrebleu.corpus_ter(hypotheses, refs_wrapped)
print(f"TER         : {ter_score.score:.2f}  (lower = better)")

### 6.2 ROUGE-1, ROUGE-2, ROUGE-L

In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

r1_f, r2_f, rl_f = [], [], []
for hyp, ref in zip(hypotheses, references):
    scores = scorer.score(ref, hyp)
    r1_f.append(scores["rouge1"].fmeasure)
    r2_f.append(scores["rouge2"].fmeasure)
    rl_f.append(scores["rougeL"].fmeasure)

print(f"ROUGE-1 F1  : {np.mean(r1_f)*100:.2f}")
print(f"ROUGE-2 F1  : {np.mean(r2_f)*100:.2f}")
print(f"ROUGE-L F1  : {np.mean(rl_f)*100:.2f}")

### 6.3 BERTScore

In [ ]:
from bert_score import score as bert_score

P, R, F1 = bert_score(
    hypotheses,
    list(references),
    lang="en",
    model_type="distilbert-base-uncased",  # lightweight; swap to roberta-large for higher quality
    verbose=True
)

print(f"BERTScore P : {P.mean().item()*100:.2f}")
print(f"BERTScore R : {R.mean().item()*100:.2f}")
print(f"BERTScore F1: {F1.mean().item()*100:.2f}")

## 7. Summary Table

In [ ]:
import pandas as pd

results = {
    "Metric"         : ["BLEU", "chrF", "chrF++", "TER ↓",
                        "ROUGE-1 F1", "ROUGE-2 F1", "ROUGE-L F1",
                        "BERTScore P", "BERTScore R", "BERTScore F1"],
    "Score"          : [
        round(bleu_score.score, 2),
        round(chrf_score.score, 2),
        round(chrfpp_score.score, 2),
        round(ter_score.score, 2),
        round(float(np.mean(r1_f))*100, 2),
        round(float(np.mean(r2_f))*100, 2),
        round(float(np.mean(rl_f))*100, 2),
        round(P.mean().item()*100, 2),
        round(R.mean().item()*100, 2),
        round(F1.mean().item()*100, 2),
    ]
}

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

## 8. Save Results

In [ ]:
results_dir = os.path.join(base_dir, "eval_results")
os.makedirs(results_dir, exist_ok=True)

# Save metric scores
metrics_path = os.path.join(results_dir, "metrics.csv")
df_results.to_csv(metrics_path, index=False)
print(f"Metrics saved → {metrics_path}")

# Save per-example predictions
df_preds = pd.DataFrame({
    "source"     : sources,
    "hypothesis" : hypotheses,
    "reference"  : references
})
preds_path = os.path.join(results_dir, "predictions.csv")
df_preds.to_csv(preds_path, index=False)
print(f"Predictions saved → {preds_path}")

## 9. Error Analysis — Worst Predictions by BLEU

In [ ]:
import sacrebleu

# Compute sentence-level BLEU for every example
sent_bleus = [
    sacrebleu.sentence_bleu(h, [r]).score
    for h, r in zip(hypotheses, references)
]

df_preds["sent_bleu"] = sent_bleus
df_preds_sorted = df_preds.sort_values("sent_bleu")

print("=== 10 Worst Translations (lowest sentence BLEU) ===")
for _, row in df_preds_sorted.head(10).iterrows():
    print(f"\nBLEU: {row['sent_bleu']:.2f}")
    print(f"  SRC : {row['source']}")
    print(f"  HYP : {row['hypothesis']}")
    print(f"  REF : {row['reference']}")

## 10. Distribution of Sentence-Level BLEU Scores

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.hist(sent_bleus, bins=40, color="steelblue", edgecolor="white", alpha=0.85)
plt.axvline(float(bleu_score.score), color="red", linestyle="--",
            label=f"Corpus BLEU = {bleu_score.score:.2f}")
plt.xlabel("Sentence BLEU", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.title("Distribution of Sentence-Level BLEU Scores\nmBART fine-tuned on PHINC (Hinglish → English)",
          fontsize=13)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(results_dir, "bleu_distribution.png"), dpi=150)
plt.show()
print("Plot saved.")

---
**Done.** All scores, predictions, and plots have been saved to `eval_results/` in your base directory.